# 🚀 HUẤN LUYỆN 4 MÔ HÌNH PHÂN ĐOẠN VẾT BỎNG (SKIN BURN SEGMENTATION)
**Mục tiêu:** Chạy 4 thực nghiệm U-Net theo bảng phân công và tự động lưu trọng số (`.pth`) + biểu đồ (`.png`) vào Google Drive.

### BƯỚC 1: Kiểm tra GPU Google Colab
*(Hãy chắc chắn đã bật GPU: Menu `Runtime` -> `Change runtime type` -> Chọn `T4 GPU`)*

In [ ]:
!nvidia-smi

### BƯỚC 2: Kết nối Google Drive & Clone Mã nguồn Dự án

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf skin-burn-analysis-system
!git clone https://github.com/Triminh15/skin-burn-analysis-system.git
%cd /content/skin-burn-analysis-system

# Sửa lỗi đường dẫn ổ D mặc định sang thư mục hiện tại '.'
!sed -i 's/"d:\/burn_analysis"/"."/g' src/vision/dataset.py src/vision/train_cls.py src/vision/postprocessing.py

### BƯỚC 3: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q segmentation-models-pytorch albumentations timm

### BƯỚC 4: Giải nén Dataset
*(Kéo thả file `data_processed.zip` (30MB) từ máy tính vào Google Drive hoặc vào cột File bên trái Colab)*

In [ ]:
import os
# Nếu bạn up data_processed.zip vào thư mục gốc của Google Drive:
if os.path.exists('/content/drive/MyDrive/data_processed.zip'):
    !unzip -q /content/drive/MyDrive/data_processed.zip -d /content/skin-burn-analysis-system/
# Hoặc nếu bạn kéo thả trực tiếp vào thư mục /content/ trên Colab:
elif os.path.exists('/content/data_processed.zip'):
    !unzip -q /content/data_processed.zip -d /content/skin-burn-analysis-system/
else:
    print("⚠️ Hãy upload file data_processed.zip lên Drive hoặc cột Files của Colab trước!")

# Kiểm tra số lượng ảnh và mask
!ls -1 data/processed/images | wc -l
!ls -1 data/processed/masks | wc -l

## 🧪 BƯỚC 5: KHỞI CHẠY 4 THỰC NGHIỆM SEGMENTATION

### 🔹 Run 1: Baseline `U-Net (ResNet-34) + ComboLoss`

In [ ]:
!python src/vision/train_seg.py \
    --arch unet \
    --backbone resnet34 \
    --loss combo \
    --epochs 35 \
    --batch-size 16 \
    --run-name unet_resnet34_combo

### 🔹 Run 2: Ablation Loss `U-Net (ResNet-34) + Focal Tversky Loss`

In [ ]:
!python src/vision/train_seg.py \
    --arch unet \
    --backbone resnet34 \
    --loss focal_tversky \
    --epochs 35 \
    --batch-size 16 \
    --run-name unet_resnet34_focal_tversky

### 🔹 Run 3: Ablation Backbone `U-Net (ConvNeXt-Tiny) + Focal Tversky Loss`

In [ ]:
!python src/vision/train_seg.py \
    --arch unet \
    --backbone tu-convnext_tiny \
    --loss focal_tversky \
    --epochs 35 \
    --batch-size 8 \
    --run-name unet_convnext_focal_tversky

### 🔹 Run 4: SOTA `Attention U-Net (ConvNeXt-Tiny) + Focal Tversky Loss`

In [ ]:
!python src/vision/train_seg.py \
    --arch unet \
    --backbone tu-convnext_tiny \
    --attention scse \
    --loss focal_tversky \
    --epochs 35 \
    --batch-size 8 \
    --run-name att_unet_convnext_focal_tversky

### BƯỚC 6: Tự động gom Trọng số & Biểu đồ lưu vào Google Drive
*(Trọng số và hình ảnh sẽ được copy ngay vào Google Drive của bạn)*

In [ ]:
import os
drive_save_dir = '/content/drive/MyDrive/Do_An_Skin_Burn_Checkpoints'
os.makedirs(drive_save_dir, exist_ok=True)

# Nén toàn bộ kết quả trọng số và biểu đồ
!zip -r /content/train_results.zip outputs/checkpoints outputs/figures

# Sao chép sang Drive
!cp /content/train_results.zip {drive_save_dir}/
!cp -r outputs/checkpoints {drive_save_dir}/
!cp -r outputs/figures {drive_save_dir}/

print(f"✅ ĐÃ SAO CHÉP TOÀN BỘ KẾT QUẢ VÀO GOOGLE DRIVE TẠI: {drive_save_dir}")